# Deliverable 4 — Paso 3: Entrenamiento QLoRA del Agente Redactor (`Qwen/Qwen3-4B`)

Este notebook ejecuta de forma reproducible el script [`Deliverable 4/scripts/entrenar_redactor.py`](scripts/entrenar_redactor.py) sobre GPU en Google Colab (T4 16 GB o superior), usando el dataset curado (`180` entrenamiento / `30` validación) y el mismo formato exacto del prompt `03_redactor.txt` (`enable_thinking=False`).

**Flujo completo:**
1. Verificar GPU y memoria VRAM disponibles.
2. Instalar dependencias (`peft`, `bitsandbytes`, `transformers>=4.51`, `accelerate`, `huggingface_hub`, `matplotlib`).
3. Cargar `Deliverable 4/` desde un ZIP local o desde GitHub (y respaldar en Google Drive).
4. Ejecutar el entrenamiento QLoRA (`completion-only loss`), generar `training_metrics.json`, `loss_curve.png/.pdf` y comparar `base` vs `fine-tuned` sobre validación (`smoke_test_val.json`).
5. Publicar el adaptador LoRA con acceso abierto en **Hugging Face** y comprobar que descarga y carga limpio con `PeftModel.from_pretrained`.
6. Descargar `D4_resultados_paso3.zip` para incorporarlo al repositorio antes del Paso 4 (evaluación completa sobre las 75 preguntas).

## Paso 0 — Revisar GPU y entorno
En Colab selecciona **Entorno de ejecución -> Cambiar tipo de entorno -> GPU (T4 / L4 / A100)**.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    nombre = torch.cuda.get_device_name(0)
    vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
    cap = torch.cuda.get_device_capability(0)
    bf16 = cap[0] >= 8 and torch.cuda.is_bf16_supported()
    print(f"GPU: {nombre} (sm_{cap[0]}{cap[1]}) | VRAM: {vram} GB | bf16 nativo: {bf16}")
else:
    raise RuntimeError("No hay GPU activa. Cambia el entorno a GPU en Colab antes de continuar.")

## Paso 1 — Instalar dependencias
No modificamos `pandas` ni `torch` preinstalados de Colab para evitar incompatibilidades del entorno.

In [ ]:
%pip install -q -U "transformers>=4.51" peft bitsandbytes accelerate "huggingface-hub>=0.30,<2.0" matplotlib
print("Dependencias listas.")

## Paso 2 — Cargar el proyecto (`Deliverable 4/`)
- Usa `MODO_FUENTE = "zip"` para subir un `.zip` que contenga `Deliverable 4/` (por ejemplo comprimido desde tu copia local).
- O cambia a `MODO_FUENTE = "git"` si ya hiciste `git push` de la rama `deliverable-4` en GitHub.

In [ ]:
import hashlib
import shutil
import subprocess
import tempfile
import zipfile
from pathlib import Path

MODO_FUENTE = "git"  # "git" o "zip"
REPO_URL = "https://github.com/Pacortes2021/GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-1.git"
RAMA_GIT = "deliverable-4"

WORKDIR = Path("/content/d4_work")
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
WORKDIR.mkdir(parents=True, exist_ok=True)

if MODO_FUENTE == "git":
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", RAMA_GIT, REPO_URL, str(WORKDIR)],
        check=True,
    )
elif MODO_FUENTE == "zip":
    from google.colab import files

    subidos = files.upload()
    zips = [n for n in subidos if n.lower().endswith(".zip")]
    assert len(zips) == 1, "Sube exactamente un archivo .zip con Deliverable 4."
    with tempfile.TemporaryDirectory() as tmp:
        tmp_path = Path(tmp).resolve()
        zip_path = tmp_path / "d4.zip"
        zip_path.write_bytes(subidos[zips[0]])
        with zipfile.ZipFile(zip_path) as zf:
            for info in zf.infolist():
                dest = (WORKDIR / info.filename).resolve()
                if not dest.is_relative_to(WORKDIR.resolve()):
                    raise ValueError(f"Ruta insegura en ZIP: {info.filename}")
            zf.extractall(WORKDIR)
else:
    raise ValueError("MODO_FUENTE debe ser 'zip' o 'git'.")

candidatos = [
    p for p in WORKDIR.rglob("Deliverable 4/scripts/entrenar_redactor.py")
    if "__MACOSX" not in p.parts
]
assert len(candidatos) == 1, f"No se encontró estructura única Deliverable 4/: {candidatos}"
D4_DIR = candidatos[0].parents[1]
SCRIPT_ENTRENAR = D4_DIR / "scripts" / "entrenar_redactor.py"
TRAIN_JSONL = D4_DIR / "datos" / "redactor_train.jsonl"
VAL_JSONL = D4_DIR / "datos" / "redactor_val.jsonl"

for p in (SCRIPT_ENTRENAR, TRAIN_JSONL, VAL_JSONL):
    assert p.is_file(), f"Falta archivo requerido: {p}"

n_train = sum(1 for l in TRAIN_JSONL.read_text(encoding="utf-8").splitlines() if l.strip())
n_val = sum(1 for l in VAL_JSONL.read_text(encoding="utf-8").splitlines() if l.strip())
print("D4_DIR:", D4_DIR)
print(f"Dataset verificado: {n_train} train / {n_val} val")
print("SHA-256 train:", hashlib.sha256(TRAIN_JSONL.read_bytes()).hexdigest()[:16])
print("SHA-256 val  :", hashlib.sha256(VAL_JSONL.read_bytes()).hexdigest()[:16])

## Paso 3 — Configurar respaldo en Drive y ejecutar QLoRA
Guardamos el adaptador, métricas, curvas y smoke test en Google Drive (y localmente en `/content`) para que no se pierdan si Colab cierra la sesión.

Hiperparámetros por defecto (verificados para que el 100 % de los ejemplos quepa sin truncamiento en T4 16 GB):
- `max_seq_length = 4096` (máximo real medido en nuestro dataset: `3.407` tokens)
- `LoRA`: `r=16`, `alpha=32`, `dropout=0.05` en `q/k/v/o_proj` + `gate/up/down_proj`
- `epochs = 3.0`, `batch_size = 1`, `grad_accum = 6` (`30` pasos/época, `90` pasos totales)
- `eval_steps = 15`, `load_best_model_at_end = True` (incluye evaluación inicial en `step 0`)

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import subprocess

USAR_DRIVE = True
STAMP = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

if USAR_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT_DIR = Path(f"/content/drive/MyDrive/D4_redactor_runs/run_{STAMP}")
else:
    OUT_DIR = Path(f"/content/d4_run_{STAMP}")

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Carpeta de salida:", OUT_DIR)

cmd = [
    "python3",
    str(SCRIPT_ENTRENAR),
    "--train-jsonl",
    str(TRAIN_JSONL),
    "--val-jsonl",
    str(VAL_JSONL),
    "--output-dir",
    str(OUT_DIR),
    "--max-seq-length",
    "4096",
    "--epochs",
    "3",
    "--grad-accum",
    "6",
    "--eval-steps",
    "15",
    "--save-steps",
    "15",
]
print("Ejecutando:", " ".join(cmd))
subprocess.run(cmd, check=True)

## Paso 4 — Revisar curvas de pérdida y smoke test (`Base` vs `Fine-tuned` en validación)

In [ ]:
import json
from IPython.display import Image, display

metricas = json.loads((OUT_DIR / "training_metrics.json").read_text(encoding="utf-8"))
print("Hardware:", metricas["hardware"]["gpu"], "| VRAM GB:", metricas["hardware"]["vram_gb"])
print("Parámetros entrenables:", metricas["parametros"])
print("Resultados:", json.dumps(metricas["resultados"], indent=2, ensure_ascii=False))

display(Image(filename=str(OUT_DIR / "loss_curve.png")))

smoke_path = OUT_DIR / "smoke_test_val.json"
if smoke_path.is_file():
    smoke = json.loads(smoke_path.read_text(encoding="utf-8"))
    for item in smoke:
        print("=" * 72)
        print(f"{item['id']} ({item['bloque']} | modo={item['modo']} | estado={item['estado']})")
        print("--- BASE ---\n" + item["salida_base"])
        print("--- FINE-TUNED ---\n" + item["salida_finetuned"])

## Paso 5 — Publicar adaptador en Hugging Face (acceso abierto) y verificar carga
1. Cambia `HF_REPO_ID` a tu repositorio público (por ejemplo `Pacortes2021/qwen3-4b-redactor-udec-d4`).
2. Si no guardaste `HF_TOKEN` en Colab Secrets, la celda abrirá `notebook_login()` para pegar tu token con permiso `write`.
3. Al finalizar, recargamos el adaptador directamente desde Hugging Face para confirmar que descarga e infiere correctamente.

In [ ]:
import os
import json
import sys
from pathlib import Path
import torch
from huggingface_hub import HfApi, notebook_login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

HF_REPO_ID = "Pacortes2021/qwen3-4b-redactor-udec-d4"  # <-- Ajusta si usas otro usuario/nombre

token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None

if not token:
    notebook_login()
    api = HfApi()
else:
    api = HfApi(token=token)

if str(D4_DIR / "scripts") not in sys.path:
    sys.path.insert(0, str(D4_DIR / "scripts"))
from entrenar_redactor import generar_model_card

adapter_dir = OUT_DIR / "adapter"
metricas_path = OUT_DIR / "training_metrics.json"
metricas = json.loads(metricas_path.read_text(encoding="utf-8"))
metricas["hf_repo"] = HF_REPO_ID
metricas["hf_url"] = f"https://huggingface.co/{HF_REPO_ID}"
(adapter_dir / "README.md").write_text(
    generar_model_card(metricas, hf_repo=HF_REPO_ID),
    encoding="utf-8",
)

api.create_repo(repo_id=HF_REPO_ID, repo_type="model", private=False, exist_ok=True)
commit_info = api.upload_folder(
    folder_path=str(adapter_dir),
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message=(
        f"Upload QLoRA Redactor D4 (eval_loss={metricas['resultados']['mejor_eval_loss']}" 
        f" @ step {metricas['resultados']['mejor_step']})"
    ),
)
metricas["hf_commit_url"] = getattr(commit_info, "commit_url", None)
metricas_path.write_text(json.dumps(metricas, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Publicado en:", metricas["hf_url"])

# Verificación real: descargar y cargar el adaptador desde Hugging Face
use_bf16 = torch.cuda.get_device_capability(0)[0] >= 8 and torch.cuda.is_bf16_supported()
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_use_double_quant=True,
)
tok_hub = AutoTokenizer.from_pretrained(metricas["modelo_base"])
base_hub = AutoModelForCausalLM.from_pretrained(
    metricas["modelo_base"], quantization_config=bnb, device_map="auto"
)
model_hub = PeftModel.from_pretrained(base_hub, HF_REPO_ID)
model_hub.eval()
from entrenar_redactor import cargar_jsonl, formatear_prompt_usuario
val0 = cargar_jsonl(VAL_JSONL)[0]
prompt0 = formatear_prompt_usuario(tok_hub, val0["messages"][0]["content"])
inp0 = tok_hub(prompt0, return_tensors="pt").to(model_hub.device)
with torch.no_grad():
    gen0 = model_hub.generate(**inp0, max_new_tokens=450, do_sample=False, pad_token_id=tok_hub.eos_token_id)
out0 = tok_hub.decode(gen0[0][inp0["input_ids"].shape[1]:], skip_special_tokens=True).strip()
print("Verificación OK: adaptador descargado y acoplado desde", HF_REPO_ID)
print("Muestra generada (val[0]):\n" + out0[:600])

## Paso 6 — Descargar artefactos livianos para el repositorio local
Empaqueta `training_metrics.json`, `loss_curve.png`, `loss_curve.pdf`, `smoke_test_val.json` y `adapter/README.md` (sin duplicar los pesos pesados en Git; los pesos quedan públicos en Hugging Face).

In [ ]:
import zipfile
from google.colab import files

bundle_zip = OUT_DIR / "D4_resultados_paso3.zip"
archivos = [
    OUT_DIR / "training_metrics.json",
    OUT_DIR / "loss_curve.png",
    OUT_DIR / "loss_curve.pdf",
    OUT_DIR / "smoke_test_val.json",
    OUT_DIR / "adapter" / "README.md",
]
with zipfile.ZipFile(bundle_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in archivos:
        if f.is_file():
            zf.write(f, arcname=str(f.relative_to(OUT_DIR)))

print("Paquete generado:", bundle_zip)
files.download(str(bundle_zip))